In [1]:
import glob
import pandas as pd
import os
import numpy as np

In [37]:
network_data = pd.read_csv('./newDataset/balanced_network_data.csv')
print(network_data.shape)

(6870587, 10)


In [38]:
benign = network_data[network_data['Label'] == 'Benign']
malicious = network_data[network_data['Label'] != 'Benign']
print("Benign samples:", benign.shape)
print("Malicious samples:", malicious.shape)

Benign samples: (4122352, 10)
Malicious samples: (2748235, 10)


Final cleaning

In [23]:
# Drop duplicates if any
network_data = network_data.drop_duplicates()

In [24]:
print("After dropping duplicates:", network_data.shape)

After dropping duplicates: (837128, 10)


In [25]:
benign = network_data[network_data['Label'] == 'Benign']
malicious = network_data[network_data['Label'] != 'Benign']
print("Benign samples:", benign.shape)
print("Malicious samples:", malicious.shape)

Benign samples: (572283, 10)
Malicious samples: (264845, 10)


In [39]:
# Handle Inf/-Inf (common in this dataset)
network_data = network_data.replace([float('inf'), -float('inf')], float('nan'))
network_data = network_data.dropna()

In [40]:
print("After dropping NAN:", network_data.shape)

After dropping NAN: (6870587, 10)


Scaling

In [41]:
from sklearn.preprocessing import StandardScaler
import numpy as np

benign = network_data[network_data['Label'] == 'Benign']
malicious = network_data[network_data['Label'] != 'Benign']

#  resolve this issue:could not convert string to float: 'Benign'
benign = benign.drop(columns=['Label'])
malicious = malicious.drop(columns=['Label'])

scaler = StandardScaler()
X_benign_scaled = scaler.fit_transform(benign)
X_malicious_scaled = scaler.transform(malicious)

In [42]:
from sklearn.model_selection import train_test_split

X_train, X_val = train_test_split(
    X_benign_scaled,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.ensemble import IsolationForest

iso_forest = IsolationForest(
    n_estimators=300,
    max_samples="auto",
    contamination="auto",   # we will set threshold manually
    max_features=1.0,
    bootstrap=False,
    n_jobs=-1,
    random_state=42
)

iso_forest.fit(X_train)

,n_estimators,300
,max_samples,'auto'
,contamination,'auto'
,max_features,1.0
,bootstrap,False
,n_jobs,-1
,random_state,42
,verbose,0
,warm_start,False


In [44]:
import numpy as np

val_scores = iso_forest.decision_function(X_val)
attack_scores = iso_forest.decision_function(X_malicious_scaled)

threshold = np.percentile(val_scores, 5)  # 5% false positive rate
print("Anomaly detection threshold:", threshold)

val_predictions = (val_scores < threshold).astype(int)  # 1 for anomaly, 0 for normal
attack_predictions = (attack_scores < threshold).astype(int)  # 1 for anomaly, 0 for normal

Anomaly detection threshold: -0.0465669512817366


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_val_true = np.zeros_like(val_predictions)  # All benign
y_attack_true = np.ones_like(attack_predictions)  # All malicious

print("Validation Set Performance:")
print(classification_report(y_val_true, val_predictions))
print("Confusion Matrix:\n", confusion_matrix(y_val_true, val_predictions))

print("Attack Set Performance:")
print(classification_report(y_attack_true, attack_predictions))
print("Confusion Matrix:\n", confusion_matrix(y_attack_true, attack_predictions))

Validation Set Performance:
              precision    recall  f1-score   support

           0       1.00      0.95      0.97    824471
           1       0.00      0.00      0.00         0

    accuracy                           0.95    824471
   macro avg       0.50      0.48      0.49    824471
weighted avg       1.00      0.95      0.97    824471

Confusion Matrix:
 [[783325  41146]
 [     0      0]]
Attack Set Performance:


c:\Users\vithu\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\vithu\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\vithu\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metri

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         0
           1       1.00      0.04      0.08   2748235

    accuracy                           0.04   2748235
   macro avg       0.50      0.02      0.04   2748235
weighted avg       1.00      0.04      0.08   2748235

Confusion Matrix:
 [[      0       0]
 [2639162  109073]]


c:\Users\vithu\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


### Autoencoder

In [2]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.callbacks import EarlyStopping

input_dim = X_train.shape[1]

input_layer = Input(shape=(input_dim,))

# Encoder
x = Dense(128, activation="relu")(input_layer)
x = Dense(64, activation="relu")(x)
x = Dense(32, activation="relu")(x)

# Bottleneck
bottleneck = Dense(16, activation="relu")(x)

# Decoder
x = Dense(32, activation="relu")(bottleneck)
x = Dense(64, activation="relu")(x)
x = Dense(128, activation="relu")(x)

output_layer = Dense(input_dim, activation="linear")(x)

autoencoder = Model(inputs=input_layer, outputs=output_layer)

autoencoder.compile(
    optimizer="adam",
    loss="mse"
)

autoencoder.summary()


c:\Users\vithu\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


NameError: name 'X_train' is not defined

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

history = autoencoder.fit(
    X_train,
    X_train,              # input == output
    epochs=100,
    batch_size=1024,
    validation_data=(X_val, X_val),
    callbacks=[early_stop],
    shuffle=True,
    verbose=1
)


In [ ]:
import numpy as np

def reconstruction_error(model, X):
    X_pred = model.predict(X, verbose=0)
    return np.mean(np.square(X - X_pred), axis=1)

val_errors = reconstruction_error(autoencoder, X_val)
attack_errors = reconstruction_error(autoencoder, X_malicious_scaled)


In [ ]:
FP_RATE = 0.01   # allow 1% benign false positives

threshold = np.percentile(val_errors, 100 * (1 - FP_RATE))
print("Reconstruction error threshold:", threshold)


In [ ]:
from sklearn.metrics import confusion_matrix

# False Positive Rate
fp_rate = val_pred.mean()

# Attack Recall
attack_recall = attack_pred.mean()

print(f"False Positive Rate (Benign): {fp_rate:.4f}")
print(f"Attack Recall: {attack_recall:.4f}")

print("Benign Confusion Matrix:")
print(confusion_matrix(np.zeros_like(val_pred), val_pred))

print("Attack Confusion Matrix:")
print(confusion_matrix(np.ones_like(attack_pred), attack_pred))
